# Week 7: Prompt Engineering

**Run this on a GPU runtime** (Colab: Runtime > Change runtime type > T4 GPU). The first run downloads a 1.5B parameter model, about 3 GB.

At the end of Week 5 we trained `TextClassifier` on 3,200 labelled Amazon reviews and asked it about a review it had never seen:

```
'not bad at all, actually pretty good'   ->  negative
```

Confidently wrong. Our model had never learned that "not bad" means something close to "good".

Today we ask the same question to a model that is **already trained**, on trillions of tokens, and we train nothing. The only thing we change is the text we feed in, **the prompt**. One review, start to finish:

1. Load a chat model and see what it actually reads
2. Ask naively
3. Ask better: instruction, output indicator, examples
4. Look inside: turn the model's next-token probabilities into a Week 5 style prediction
5. Rerun the Week 5 test sentences

## 1. Loading a chat model

**Qwen2.5-1.5B-Instruct** was pretrained on next-token prediction, the same objective as our Week 4 `TinyTransformerLM` but at enormous scale, and then fine-tuned on conversations so that it answers instead of just continuing text.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
model.eval()

# Always pick the most likely token (greedy decoding, Week 3), so every run gives the same answer
# and every difference we see comes from the prompt.
gc = model.generation_config
gc.do_sample, gc.temperature, gc.top_p, gc.top_k, gc.repetition_penalty = False, 1.0, 1.0, None, 1.0

review = "not bad at all, actually pretty good"

### What the model actually reads

We write messages, but the model, just like `TinyTransformerLM`, only ever sees **one string of tokens**. The **chat template** turns a list of messages into that string, using special tokens to mark each turn and who is speaking (the **role**).

In [ ]:
messages = [{"role": "user", "content": f"What is the sentiment of this review?\n\n{review}"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(text)

- `<|im_start|>` opens a turn and names the role, `<|im_end|>` closes it. Each is a single token in the vocabulary (Week 2).
- We never wrote a **system** turn; Qwen's template adds a default one. The system turn holds standing rules, the **user** turn holds the request.
- The string ends with an open `<|im_start|>assistant` turn. The most likely continuation of that string is an assistant's reply, so plain next-token prediction now *answers*.

A small helper does this every time: build the string, generate, and keep only the new tokens.

In [ ]:
def chat(messages, max_new_tokens=60):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0, inputs["input_ids"].shape[1]:]          # drop the prompt, keep the reply
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 2. Ask naively

In [ ]:
print(chat(messages))

The model probably reads the review correctly, which Week 5's classifier could not. But look at the **form** of the answer: a sentence, maybe a paragraph. A person can read it. A program that needs a label cannot use it without digging the word out, and the wording changes from review to review.

## 3. Ask better

### 3.1 Instruction and output indicator

A prompt is built from parts. The two we need:

- **Instruction:** the task, stated precisely.
- **Output indicator:** what the answer must look like, plus a cue that the answer comes next (`Sentiment:`).

Same model, same review, only the text changes:

In [ ]:
INSTRUCTION = ("Classify the sentiment of the product review as positive or negative. "
               "Answer with exactly one word: positive or negative.")

def prompt(review):
    return [{"role": "user", "content": f"{INSTRUCTION}\n\nReview: {review}\nSentiment:"}]

print(chat(prompt(review)))

Bigger tasks need more parts, such as persona, context, audience and tone (see slides). The method is always the same: change one part, rerun, compare.

### In-context learning

We can also put **examples** in the prompt. This is called **in-context learning**: zero-shot has no examples (what we just did), one-shot has one, few-shot has two or more. In a chat model each example is an earlier user turn followed by the correct assistant reply.

In [ ]:
examples = [("Arrived broken and support never answered.", "negative"),
            ("Exactly what I needed, works perfectly.", "positive")]

def few_shot_prompt(review):
    msgs = []
    for ex_review, ex_label in examples:
        msgs += prompt(ex_review)
        msgs.append({"role": "assistant", "content": ex_label})
    return msgs + prompt(review)

for m in few_shot_prompt(review):
    print(f"{m['role']:9}| {m['content'][-60:]!r}")
print("\nreply:", chat(few_shot_prompt(review)))

No weights changed. The model "learned" the format only for this one prompt; the next prompt starts from nothing. That is the key difference from Week 5, where learning meant updating weights with backpropagation.

Both prompts **ask** for one word. Neither **guarantees** it: on some other review the model may still reply "Positive." or "The review is positive". Section 4 fixes that for good.

## 4. Look inside: from next-token probabilities to a prediction

When the model writes its answer, it does exactly what we did in Weeks 3 and 4: the last position's logits go through a softmax over the **whole vocabulary**, and the top token is picked. Here are the top 5 tokens right after `Sentiment:`.

In [ ]:
text = tokenizer.apply_chat_template(prompt(review), tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    logits = model(**inputs).logits[0, -1].float()       # one score per vocabulary token
probs = torch.softmax(logits, dim=-1)

print(f"vocabulary size: {len(probs):,}\n")
for p, i in zip(*torch.topk(probs, 5)):
    print(f"{tokenizer.decode([int(i)])!r:12} logit={logits[i]:6.2f}   P={p:.4f}")

A classifier only cares about two of those ~150,000 tokens. So we do what Week 4's causal mask did: set every other logit to $-\infty$ and take the softmax again. Everything else gets probability 0, and the two labels share all of it. The output now has exactly the shape of Week 5's `TextClassifier`: two probabilities that add up to 1.

In [ ]:
label_ids = [tokenizer.encode(w, add_special_tokens=False)[0] for w in ["negative", "positive"]]
assert label_ids[0] != label_ids[1], "the two labels must start with different tokens"

mask = torch.full_like(logits, float("-inf"))
mask[label_ids] = 0.0
two_way = torch.softmax(logits + mask, dim=-1)[label_ids]

print(f"logit(negative) = {logits[label_ids[0]]:.2f}   logit(positive) = {logits[label_ids[1]]:.2f}")
print(f"P(negative) = {two_way[0]:.2f}   P(positive) = {two_way[1]:.2f}")

You can check it with the two printed logits: $P(\text{positive}) = \frac{e^{\,\text{logit(pos)}}}{e^{\,\text{logit(neg)}} + e^{\,\text{logit(pos)}}}$, the same two-class softmax as Week 5.

This trick has a name, **constrained sampling**: only allowed tokens can be picked, so the answer is *always* a valid label. Libraries apply the same mask at every step to force valid JSON or other formats (slides).

Wrapped into the same `predict()` interface as Week 5:

In [ ]:
def predict(review):
    text = tokenizer.apply_chat_template(prompt(review), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1].float()
    mask = torch.full_like(logits, float("-inf"))
    mask[label_ids] = 0.0
    probs = torch.softmax(logits + mask, dim=-1)[label_ids]   # [P(negative), P(positive)]
    label = "positive" if probs[1] > probs[0] else "negative"
    return label, probs.tolist()

label, p = predict(review)
print(f"{review!r:55s} -> {label:8s} (neg={p[0]:.2f}, pos={p[1]:.2f})")

## 5. The Week 5 test sentences, again

The same five sentences we gave `TextClassifier` at the end of Week 5. The middle column is what Week 5's model said.

In [ ]:
week5 = {   # TextClassifier's predictions from the Week 5 notebook
    "not really great": "positive",
    "This phone completely changed how I take photos, I love it": "positive",
    "Terrible battery life, would not buy again": "negative",
    "It's fine I guess, does what it says": "negative",
    "not bad at all, actually pretty good": "negative",
}
print(f"{'review':62} {'Week 5':9} {'Week 7 (prompt)'}")
for r, old in week5.items():
    label, p = predict(r)
    print(f"{r!r:62} {old:9} {label:8} (neg={p[0]:.2f}, pos={p[1]:.2f})")

**Try it:** type your own review, especially a tricky one (sarcasm, mixed feelings, double negation).

In [ ]:
my_review = "I wanted to hate it, but I can't."
label, p = predict(my_review)
print(f"{my_review!r} -> {label} (neg={p[0]:.2f}, pos={p[1]:.2f})")

### What to take away

- **Week 5:** we built the model, trained it on 3,200 labels, and it still failed on "not bad".
- **Week 7:** we trained nothing. A pretrained model plus a prompt plus a two-token mask became a classifier. The credit goes to pretraining on trillions of tokens; the prompt is how we reach that knowledge.
- The mask guarantees a **valid** label, not a **correct** one. The model can still be wrong, and with sarcasm or mixed reviews it will be.

## Exercises

Load the Week 5 dataset and pick 200 reviews (100 positive, 100 negative).

**Part A.** Run `predict()` on all 200 and report accuracy. Compare it with the roughly 80% Week 5's `TextClassifier` reached.

**Part B.** Write `predict_few_shot()` that uses `few_shot_prompt` instead of `prompt`, run it on the same 200, and compare. Then show two reviews the model gets wrong, and explain in a sentence each why you think it failed.